# NBA RAG Pipeline — Starter Notebook

Goal for this notebook (Week 1 of the roadmap): get the full RAG loop working end to end,
even if it's ugly. Ingest stats -> chunk -> embed -> store -> retrieve -> generate a cited answer.

Pipeline stages:
1. **Ingest** — pull real NBA stats (game logs, season averages) via `nba_api` (free, no key needed)
2. **Chunk** — turn structured stats into short text chunks, grouped by player + time window
3. **Embed** — turn each chunk into a vector using Voyage AI (Anthropic's recommended embedding partner — Claude itself has no embeddings endpoint)
4. **Store** — save chunks + embeddings in Supabase Postgres (pgvector extension)
5. **Retrieve** — given a question, embed it and pull the most similar stored chunks
6. **Generate** — hand the retrieved chunks to Claude and ask it to answer *using only that context*, citing which chunk it used

This version intentionally skips LangChain abstractions so every step is visible. Once this works,
wrap it in LangChain / FastAPI for Week 2 of the roadmap.


## 0. Setup

Install dependencies (run once), then set your API keys as environment variables rather than
hardcoding them in the notebook.

Keys you'll need:
- `ANTHROPIC_API_KEY` — Claude API
- `VOYAGE_API_KEY` — Voyage AI embeddings ([dashboard.voyageai.com](https://dashboard.voyageai.com))
- `SUPABASE_DB_URL` — your Supabase project's Postgres connection string (Project Settings -> Database -> Connection string, use the "URI" one). Must have the `pgvector` extension enabled (Database -> Extensions -> vector).


In [ ]:
# Run once. If you're using a venv/conda env, run this in a terminal instead.
# !pip install nba_api voyageai anthropic psycopg2-binary pgvector python-dotenv


In [ ]:
import os
import time
from datetime import datetime, timedelta

from dotenv import load_dotenv
load_dotenv()  # reads a local .env file if you have one — don't commit it

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
SUPABASE_DB_URL = os.environ["SUPABASE_DB_URL"]


## 1. Ingest — pull real stats

Start small: a handful of players you can sanity-check by eye. `nba_api` wraps stats.nba.com and is
free, but it's an unofficial wrapper — expect occasional flaky requests, so we add a small retry/sleep.

Swap in BallDontLie later if you want injury/news endpoints (`ALL-STAR` tier, $9.99/mo covers that).


In [ ]:
from nba_api.stats.static import players as static_players
from nba_api.stats.endpoints import playergamelog

# Start with a small, well-known set so you can eyeball correctness
WATCHLIST = ["Jayson Tatum", "Devin Booker", "Anthony Edwards"]

def get_player_id(full_name: str) -> int:
    matches = static_players.find_players_by_full_name(full_name)
    if not matches:
        raise ValueError(f"No player found for {full_name!r}")
    return matches[0]["id"]

def fetch_recent_gamelog(player_name: str, season: str = "2025-26", retries: int = 3):
    player_id = get_player_id(player_name)
    for attempt in range(retries):
        try:
            log = playergamelog.PlayerGameLog(player_id=player_id, season=season)
            return log.get_data_frames()[0]  # one row per game, most recent first
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(1.5)

gamelogs = {}
for name in WATCHLIST:
    gamelogs[name] = fetch_recent_gamelog(name)
    time.sleep(0.6)  # be polite to the unofficial API

gamelogs[WATCHLIST[0]].head()


## 2. Chunk — group stats into text, by player + time window

RAG retrieval works on chunks of text, not raw dataframes. The chunking decision here is:
**one chunk per player per rolling window** (e.g. last 5 games, last 14 days). This keeps chunks
small and specific, so retrieval can pull "just Tatum's last 2 weeks" instead of his whole season.

Each chunk is plain English summarizing the numbers — that's what gets embedded and what Claude
reads later.


In [ ]:
def summarize_recent_games(player_name: str, df, n_games: int = 5) -> str:
    recent = df.head(n_games)
    avg_pts = recent["PTS"].mean()
    avg_reb = recent["REB"].mean()
    avg_ast = recent["AST"].mean()
    fg_pct = recent["FG_PCT"].mean()
    dates = f"{recent['GAME_DATE'].iloc[-1]} to {recent['GAME_DATE'].iloc[0]}"

    chunk_text = (
        f"{player_name} — last {n_games} games ({dates}): "
        f"averaging {avg_pts:.1f} pts, {avg_reb:.1f} reb, {avg_ast:.1f} ast "
        f"on {fg_pct*100:.1f}% shooting from the field. "
        f"Game-by-game: " + "; ".join(
            f"{row.GAME_DATE} vs {row.MATCHUP.split(' ')[-1]}: "
            f"{row.PTS} pts/{row.REB} reb/{row.AST} ast"
            for row in recent.itertuples()
        )
    )
    return chunk_text

chunks = []  # list of {"id": ..., "player": ..., "text": ...}
for name, df in gamelogs.items():
    text = summarize_recent_games(name, df, n_games=5)
    chunks.append({
        "id": f"{name.replace(' ', '_').lower()}_last5",
        "player": name,
        "text": text,
    })

for c in chunks:
    print(c["id"], "->", c["text"][:120], "...\n")


## 3. Embed — turn each chunk into a vector

Claude doesn't have an embeddings endpoint, which is why Voyage AI (Anthropic's recommended partner)
is doing this step. `voyage-3` or `voyage-3-lite` are reasonable defaults for cost/quality here.


In [ ]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

def embed_texts(texts: list[str], input_type: str = "document") -> list[list[float]]:
    # input_type differs for what you're storing ("document") vs what you're searching with ("query")
    result = vo.embed(texts, model="voyage-3", input_type=input_type)
    return result.embeddings

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_texts(chunk_texts, input_type="document")

print(f"Embedded {len(chunk_embeddings)} chunks, dimension = {len(chunk_embeddings[0])}")


## 4. Store — Supabase Postgres + pgvector

Run this SQL once in the Supabase SQL editor before this cell works:

```sql
create extension if not exists vector;

create table if not exists nba_chunks (
    id text primary key,
    player text,
    content text,
    embedding vector(1024)  -- match voyage-3's output dimension
);
```

Then this cell inserts (or updates) each chunk + its embedding.


In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(SUPABASE_DB_URL)
register_vector(conn)
cur = conn.cursor()

for c, emb in zip(chunks, chunk_embeddings):
    cur.execute(
        """
        insert into nba_chunks (id, player, content, embedding)
        values (%s, %s, %s, %s)
        on conflict (id) do update
        set content = excluded.content, embedding = excluded.embedding
        """,
        (c["id"], c["player"], c["text"], emb),
    )

conn.commit()
print(f"Stored {len(chunks)} chunks in Supabase.")


## 5. Retrieve — semantic search over stored chunks

Start with pure vector similarity (cosine distance, pgvector's `<=>` operator). If you notice
retrieval missing exact player names or stat keywords once you have more chunks, that's the signal
to add the keyword (`tsvector`) half of hybrid search — see Supabase's hybrid search docs.


In [ ]:
def retrieve(question: str, k: int = 3):
    query_embedding = embed_texts([question], input_type="query")[0]

    cur.execute(
        """
        select id, player, content, embedding <=> %s as distance
        from nba_chunks
        order by embedding <=> %s
        limit %s
        """,
        (query_embedding, query_embedding, k),
    )
    rows = cur.fetchall()
    return [{"id": r[0], "player": r[1], "content": r[2], "distance": r[3]} for r in rows]

# Sanity check
retrieve("How has Tatum looked lately?")


## 6. Generate — Claude answers using only the retrieved context

The key RAG discipline: tell Claude explicitly to only use the provided context and to cite which
chunk(s) it drew from. This is what makes the answer "grounded" instead of a hallucinated guess.


In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def answer_question(question: str, k: int = 3) -> str:
    retrieved = retrieve(question, k=k)

    context_block = "\n\n".join(
        f"[Source: {r['id']}] {r['content']}" for r in retrieved
    )

    prompt = f"""You are an NBA stats assistant. Answer the user's question using ONLY the
context provided below. If the context doesn't contain enough information to answer, say so —
do not make anything up. Cite which [Source: ...] you used.

Context:
{context_block}

Question: {question}
"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

print(answer_question("Who should I start this week, Tatum or Booker?"))


## Next steps (don't do these yet — see ROADMAP.md)

- Try 5-10 more questions by hand and note where retrieval pulls the wrong chunk, or Claude answers
  outside the given context — that's your eval test set forming naturally
- Add injury/news chunks (unstructured text) alongside these stat chunks, so questions like
  "is Tatum playing tonight" have something to retrieve
- Once this feels solid, move ingestion + retrieval + generation into FastAPI endpoints (Week 2)
- Add hybrid (keyword + semantic) search only if you actually observe retrieval misses — don't
  add it speculatively
